In [ ]:
# ===============================
# Jupyter Notebook: K线 + 交易点 + 滑动条
# ===============================

import pandas as pd
import plotly.graph_objects as go
from vnpy.trader.constant import Direction, Offset
from run_backtest import run_backtest  # 你的回测脚本

# -------------------------------
# 1️⃣ 运行回测
# -------------------------------
engine = run_backtest()       # 确保 run_backtest 返回 engine
strategy = engine.strategy

# -------------------------------
# 2️⃣ 获取长周期 BarData
# -------------------------------
bars_list = getattr(strategy, "bg_long_bars", [])
if not bars_list:
    raise RuntimeError("策略没有保存 BarData，请确保在 on_long_bar 中 append BarData")

# -------------------------------
# 3️⃣ 最近 N 根
N = 500
bars_list = bars_list[-N:]

# -------------------------------
# 4️⃣ 转 DataFrame
# -------------------------------
df = pd.DataFrame([{
    "datetime": b.datetime,
    "open": b.open_price,
    "high": b.high_price,
    "low": b.low_price,
    "close": b.close_price,
    "volume": b.volume
} for b in bars_list])

# -------------------------------
# 5️⃣ 添加交易点列
# -------------------------------
df["long_entry"] = False
df["long_exit"] = False
df["short_entry"] = False
df["short_exit"] = False

for t in engine.trades.values():
    t_dt = t.datetime
    mask = df["datetime"] <= t_dt
    if mask.any():
        idx = df[mask].index[-1]
        if t.direction == Direction.LONG and t.offset == Offset.OPEN:
            df.at[idx, "long_entry"] = True
        elif t.direction == Direction.LONG and t.offset == Offset.CLOSE:
            df.at[idx, "long_exit"] = True
        elif t.direction == Direction.SHORT and t.offset == Offset.OPEN:
            df.at[idx, "short_entry"] = True
        elif t.direction == Direction.SHORT and t.offset == Offset.CLOSE:
            df.at[idx, "short_exit"] = True

# -------------------------------
# 6️⃣ 绘制 Plotly K 线 + 交易点 + 滑动条
# -------------------------------
fig = go.Figure()

# K 线
fig.add_trace(go.Candlestick(
    x=df["datetime"],
    open=df["open"],
    high=df["high"],
    low=df["low"],
    close=df["close"],
    name="Kline"
))

# 交易点
fig.add_trace(go.Scatter(
    x=df.loc[df["long_entry"], "datetime"],
    y=df.loc[df["long_entry"], "close"],
    mode="markers",
    marker=dict(symbol="triangle-up", color="green", size=10),
    name="Long Entry"
))
fig.add_trace(go.Scatter(
    x=df.loc[df["long_exit"], "datetime"],
    y=df.loc[df["long_exit"], "close"],
    mode="markers",
    marker=dict(symbol="triangle-down", color="red", size=10),
    name="Long Exit"
))
fig.add_trace(go.Scatter(
    x=df.loc[df["short_entry"], "datetime"],
    y=df.loc[df["short_entry"], "close"],
    mode="markers",
    marker=dict(symbol="triangle-down", color="blue", size=10),
    name="Short Entry"
))
fig.add_trace(go.Scatter(
    x=df.loc[df["short_exit"], "datetime"],
    y=df.loc[df["short_exit"], "close"],
    mode="markers",
    marker=dict(symbol="triangle-up", color="orange", size=10),
    name="Short Exit"
))

# -------------------------------
# 7️⃣ 布局设置
# -------------------------------
fig.update_layout(
    title="Backtest Kline with Trades",
    xaxis_title="Datetime",
    yaxis_title="Price",
    xaxis_rangeslider_visible=True,   # 显示滑动条
    xaxis_rangeslider_thickness=0.1, # 调整滑动条厚度
    template="plotly_dark"
)

# -------------------------------
# 8️⃣ 显示
# -------------------------------
fig.show()


=== vn.py 回测脚本 ===
当前工作目录: d:\finance\data\vnpy_ctastrategy\samir
Python路径: c:\veighna_studio\python.exe
成功导入策略: LlmStrategyExact
正在加载数据...
2025-12-09 18:05:44.769942	开始加载历史数据
2025-12-09 18:05:44.769975	加载进度：# [0%]
2025-12-09 18:05:44.770299	加载进度：# [10%]
2025-12-09 18:05:44.770573	加载进度：## [20%]
2025-12-09 18:05:44.771000	加载进度：### [30%]
2025-12-09 18:05:44.771564	加载进度：#### [40%]
2025-12-09 18:05:44.772069	加载进度：##### [50%]
2025-12-09 18:05:44.772576	加载进度：###### [60%]
2025-12-09 18:05:44.773018	加载进度：####### [70%]
2025-12-09 18:05:44.773512	加载进度：######## [80%]
2025-12-09 18:05:44.774138	加载进度：######### [90%]
2025-12-09 18:05:44.774371	加载进度：########## [100%]
2025-12-09 18:05:44.774386	历史数据加载完成，数据量：296695
数据加载完成
开始回测...
2025-12-09 18:05:44.774413	策略初始化完成
2025-12-09 18:05:44.774421	开始回放历史数据
[100] 计数器 Bar: 2025-02-12 04:00:00+08:00 O:2891.57 H:2894.38 L:2883.61 C:2893.33 pos_state info: None, 0.0,0.0, 0.0, 0.0
[101] 计数器 Bar: 2025-02-12 08:00:00+08:00 O:2893.28 H:2895.66 L:2879.13 C:2882.4 pos_s